<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-14%20%E2%80%94%20Mini-ETL%20on%20Walmart%20Sample%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

The whole pipeline is written as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

Every stage is worked through below. **Run the cells, read what each one did, and change things to see what breaks.** The one thing to produce yourself is the checkpoint at the end.

In [1]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [2]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('columns that arrived as text:',
          [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])])
    print('nulls per column:')
    print(df.isnull().sum()[lambda s: s > 0].to_string())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

extracted 12 rows, 8 columns
columns that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls per column:
timestamp    1
quantity     1
exact duplicates: 1


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
1,TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
3,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
4,TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,$4.11


The report already wrote your to-do list, from the data rather than from guessing: `unit_price` arrived as **text** because of the dollar signs, two columns have nulls, and there is an exact duplicate.

One result there is worth stopping on. **`quantity` came back numeric, not text** — even though row `TXN9` literally contains the word `NULL`. `read_csv` recognises `NULL` as one of its missing-value spellings and converted it to `NaN` for you, which is why the column could stay a number. That is convenient and it is also a decision somebody else made about your data. If the file had said `n/a` or `-999` instead, the whole column would have come back as text and the fix would be yours.

Here is everything wrong with these twelve rows. Find each one in the table above before reading on — this is the part that transfers to Monday, when nobody hands you the list:

| # | Problem | Where |
|---|---|---|
| 1 | An exact duplicate row | `TXN1` appears twice |
| 2 | Dollar signs on some prices, not others | `$4.11` vs `4.11` |
| 3 | A missing quantity, written as `NULL` | `TXN9`, the flashlight |
| 4 | A negative quantity | `TXN3`, a return |
| 5 | Four spellings of two stores | `FL-105`, `fl-105`, `FL 239`, `fl-239` |
| 6 | Three timestamp formats, plus one empty | `TXN8` has no timestamp |
| 7 | Four SKUs for one product | strawberry Pop-Tarts |
| 8 | Two SKUs for another | bottled water |
| 9 | Inconsistent category case | `Breakfast` vs `breakfast` |

Nine problems in twelve rows. The full file has the same nine in three million.

### Build 1 — Transform: types and duplicates

`transform_types` returns a **new** frame and leaves the one you passed in alone. That is what `.copy()` buys, and it is why you can re-run this cell without the results drifting.

In [3]:
def transform_types(df):
    out = df.drop_duplicates().copy()

    out['unit_price'] = (out['unit_price'].astype(str)
                         .str.replace('$', '', regex=False)
                         .str.strip().astype(float))

    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    # Decision: drop the unparseable quantity and the return. We are measuring
    # units that left the shelf ahead of the storm, so a return is not that.
    out = out[out['quantity'].notna() & (out['quantity'] > 0)].copy()
    out['quantity'] = out['quantity'].astype(int)

    # FL-105 / fl-105 / 'FL 239' / fl-239 all collapse to FL-105 / FL-239
    out['store_id'] = (out['store_id'].str.upper().str.strip()
                       .str.replace(r'[^A-Z0-9]', '', regex=True)
                       .apply(lambda s: f'{s[:2]}-{s[2:]}'))

    out['category'] = out['category'].str.strip().str.title()
    out['timestamp'] = pd.to_datetime(out['timestamp'],
                                      errors='coerce', format='mixed')
    return out

typed = transform_types(raw)
print('rows:', len(raw), '->', len(typed))
print('stores:', sorted(typed['store_id'].unique()))
print('unparseable timestamps:', typed['timestamp'].isna().sum())
typed.dtypes

rows: 12 -> 9
stores: ['FL-105', 'FL-239']
unparseable timestamps: 1


,0
transaction_id,object
store_id,object
timestamp,datetime64[ns]
sku,object
product_name,object
category,object
quantity,int64
unit_price,float64


Twelve rows in, nine out. One was the duplicate, one was the `NULL` quantity, one was the return.

**Order matters, and this is the trap.** Deduplicating happens *first*, before anything else touches the frame. Clean the values first and the duplicate stops being an exact match, so `drop_duplicates()` no longer sees it and you carry a phantom sale all the way to your conclusion.

**The quantity filter is a judgment call, not a rule.** Dropping the return is right *for this question* — we are counting units that left the shelf before a storm, and a refund is not that. For a revenue report you would keep it, because the money genuinely went back. The comment in the code says which question it serves, and that is the part to copy on Monday.

**`pd.to_numeric(..., errors='coerce')` is belt and braces here.** `read_csv` already made this column numeric, so on this file it changes nothing. It stays in because the full dataset on Monday is not guaranteed to be so cooperative — one `n/a` anywhere in that column and it arrives as text, at which point `quantity * unit_price` repeats strings instead of multiplying numbers and nothing raises to tell you.

**`format='mixed'`** lets pandas parse each timestamp on its own terms, which is the only way three different formats survive in one column. The empty one becomes `NaT` — still there, still countable, which is why we print the count rather than assuming it is zero.

### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. Two are the same bottled water. One is blueberry, and it stays by itself.

In [4]:
CANON = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'WATER-24': 'WATER-24',
    'BTL-WATER-24': 'WATER-24',
}

def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
    return out

consolidated = consolidate(typed)
assert consolidated['quantity'].sum() == typed['quantity'].sum(), \
    'consolidation must not change unit totals'
print(consolidated.groupby('canonical_sku')['quantity'].sum()
      .sort_values(ascending=False))

canonical_sku
POPTART-STRAW    17
WATER-24         10
POPTART-BLUE      1
Name: quantity, dtype: int64


**`.fillna(out['sku'])` is the whole cell.** `.map()` returns `NaN` for any key that is not in the dictionary, so without that `fillna` the flashlight and the blueberry Pop-Tarts become `NaN`, drop out of every later `groupby`, and vanish from the analysis. No error, no warning — just products that quietly stop existing. Delete the `.fillna(...)` and re-run it if you want to watch that happen.

**The assertion is the safety rail.** Consolidating *renames* products; it must never change how many units you have. If that total moves, your mapping did something other than what you intended, and you want to find out here rather than in a chart.

**Why blueberry stays separate.** The Walmart finding was about strawberry specifically. Folding the flavors together would average the signal away — the same mistake as leaving the strawberry SKUs fragmented, just in the opposite direction. Consolidate what is genuinely the same product; keep apart what a buyer would order separately.

### Build 3 — Transform: derive what the analysis needs

Two new columns: `revenue`, and a `date` you can join weather onto next week.

In [5]:
def derive(df):
    out = df.copy()
    out['revenue'] = (out['quantity'] * out['unit_price']).round(2)
    out['date'] = out['timestamp'].dt.normalize()
    return out

enriched = derive(consolidated)
enriched[['transaction_id', 'store_id', 'canonical_sku',
          'quantity', 'revenue', 'date']]

,transaction_id,store_id,canonical_sku,quantity,revenue,date
0,TXN1,FL-105,POPTART-STRAW,2,8.22,2024-09-06
1,TXN2,FL-105,POPTART-STRAW,1,3.86,2024-09-06
4,TXN4,FL-239,POPTART-STRAW,3,12.33,2024-09-06
5,TXN5,FL-239,WATER-24,4,20.00,2024-09-06
6,TXN6,FL-105,WATER-24,6,30.00,2024-09-07
7,TXN7,FL-105,POPTART-STRAW,5,20.55,2024-09-07
8,TXN8,FL-239,POPTART-STRAW,2,8.22,NaT
10,TXN10,FL-239,POPTART-BLUE,1,4.11,2024-09-07
11,TXN11,FL-105,POPTART-STRAW,4,16.44,2024-09-07


`.dt.normalize()` keeps the column a real datetime and sets the clock to midnight. The alternative, `.dt.date`, gives you Python `date` objects — which look identical when printed and then refuse to join cleanly against a datetime column from another source. Next week you are joining this to weather, so it needs to stay a datetime.

`.round(2)` is there because floating-point money accumulates noise. `5 * 4.11` is not exactly `20.55` in binary, and after a few thousand rows those crumbs show up in a total somebody is going to read.

### Build 4 — Load, and verify the load

Writing without reading back is how you discover at 11pm that `if_exists='append'` doubled your table.

In [6]:
def load(df, conn, table='clean_sales'):
    df.to_sql(table, conn, index=False, if_exists='replace')
    written = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {table}', conn)['n'][0]
    assert written == len(df), f'wrote {written}, expected {len(df)}'
    print(f'loaded {written} rows into {table}')
    return written

conn = sqlite3.connect(':memory:')
load(enriched, conn)

pd.read_sql_query(
    'SELECT canonical_sku, SUM(quantity) AS units, ROUND(SUM(revenue), 2) AS revenue '
    'FROM clean_sales GROUP BY canonical_sku ORDER BY units DESC', conn)

loaded 9 rows into clean_sales


,canonical_sku,units,revenue
0,POPTART-STRAW,17,69.62
1,WATER-24,10,50.00
2,POPTART-BLUE,1,4.11


Three arguments doing real work. **`index=False`** stops pandas writing its row numbers as a column you never asked for. **`if_exists='replace'`** drops and rebuilds the table, which is what makes re-running safe; `'append'` would stack a second copy on top every time you ran the cell. And the **`assert`** reads the count back out of the database rather than trusting that the write did what it said.

That last one is the habit. A write that reports no error has still only told you that nothing *raised*. Counting the rows afterwards is how you know.

### Build 5 — run the whole thing as one pipeline

The stages wired together, then run twice against fresh connections to prove the result does not depend on how many times you ran it.

In [7]:
def run_pipeline(conn):
    df = extract()
    df = transform_types(df)
    df = consolidate(df)
    df = derive(df)
    load(df, conn)
    return df

first = run_pipeline(sqlite3.connect(':memory:'))
second = run_pipeline(sqlite3.connect(':memory:'))

assert first.shape == second.shape
assert first['revenue'].sum() == second['revenue'].sum()
print('identical across runs:', first.equals(second))

extracted 12 rows, 8 columns
columns that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls per column:
timestamp    1
quantity     1
exact duplicates: 1
loaded 9 rows into clean_sales
extracted 12 rows, 8 columns
columns that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls per column:
timestamp    1
quantity     1
exact duplicates: 1
loaded 9 rows into clean_sales
identical across runs: True


That is **idempotency**, and it is the property that separates a pipeline from a script: running it twice gives you what running it once gave you.

It holds here for reasons you can point at. Every stage takes a frame and returns a new one instead of editing in place, so nothing accumulates. The load uses `'replace'`. Nothing depends on a variable left over from a previous cell.

Notice that this is **tested, not assumed**. The two assertions are three lines, and they are the difference between believing your pipeline is repeatable and knowing it.

### Build 6 — the question the pipeline exists to answer

Did strawberry Pop-Tart sales rise from September 6 to September 7?

In [8]:
straw = enriched[enriched['canonical_sku'] == 'POPTART-STRAW']
by_day = straw.groupby(straw['date'].dt.date)['quantity'].sum()
print(by_day)
print()
print('units with no date:', int(straw.loc[straw['date'].isna(), 'quantity'].sum()))

date
2024-09-06    6
2024-09-07    9
Name: quantity, dtype: int64

units with no date: 2


Units rise from September 6 to September 7. **That is not evidence of a surge, and saying so is the correct answer.**

Two days with nothing to compare them against is not a trend. It could be a storm, or a weekend, or two people. The Walmart claim is meaningful only because it was measured against weeks of ordinary sales, and that baseline is what you get on the midterm.

Also note the units sitting on no date at all. `TXN8` has no timestamp, so its quantity is real but belongs to neither day — it is in your totals and missing from your daily breakdown. Printing that number is the difference between a result and a result you can defend. A reader who finds it themselves stops trusting the rest.

---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [9]:
# Checkpoint
rows_after_transform = len(typed)  # 9
canonical_skus = enriched['canonical_sku'].nunique()  # 3
idempotent = first['revenue'].sum() == second['revenue'].sum()  # True
blueberry_call = 'Kept separate because blueberry is a different product flavor from strawberry.'

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)

rows after transform: 9
canonical SKUs: 3
same result twice: True
blueberry: Kept separate because blueberry is a different product flavor from strawberry.
